<a href="https://colab.research.google.com/github/andres-merino/AprendizajeAutomaticoInicial-05-N0105/blob/main/2-Notebooks/02-Metricas-Distancia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="border: none; border-collapse: collapse;">
    <tr>
        <td style="width: 20%; vertical-align: middle; padding-right: 10px;">
            <img src="https://i.imgur.com/nt7hloA.png" width="100">
        </td>
        <td style="width: 2px; text-align: center;">
            <font color="#0030A1" size="7">|</font><br>
            <font color="#0030A1" size="7">|</font>
        </td>
        <td>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="5">
                <b>Facultad de Ciencias Exactas, Naturales y Ambientales</b>
            </font> </p>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="4">
                Aprendizaje Automático Inicial &bull; Métricas de distancia
            </font></p>
            <p style="font-style: oblique;"><font color="#0030A1" size="3">
                Andrés Merino &bull; Periodo 2026-2
            </font></p>
        </td>  
    </tr>
</table>

## Introducción
Comparar distancias euclidiana, Manhattan, Minkowski y Mahalanobis, y explicar cómo las unidades afectan la noción de cercanía. Antes de ejecutar cada ejemplo, anticipa el resultado. Abre el cuaderno en Colab y guarda una copia para trabajar.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Distancias entre dos puntos
Para $p\geq 1$, $d_p(x,y)=(\sum_i |x_i-y_i|^p)^{1/p}$. Manhattan corresponde a $p=1$ y euclidiana a $p=2$.
Calcula primero a mano las distancias entre $(1,2)$ y $(4,6)$. ¿Qué valores esperas para $p=1$ y $p=2$?


In [ ]:
x = np.array([1., 2.])
y = np.array([4., 6.])
distancias = {f"Minkowski p={p}": np.sum(np.abs(x-y)**p)**(1/p) for p in [1, 2, 3]}
pd.Series(distancias, name="Distancia")


## Efecto de las escalas
Una vivienda de referencia tiene 2 habitaciones y 100 m². A y B son dos alternativas. ¿Cuál es más cercana? Compara las distancias al expresar el área en m² y en miles de m². Cambian las unidades, no las viviendas.


In [ ]:
viviendas = pd.DataFrame({"habitaciones": [2., 3., 2.], "area_m2": [100., 100., 120.]}, index=["Referencia", "A", "B"])
original = viviendas.to_numpy()
otras_unidades = original / np.array([1., 1000.])
comparacion = pd.DataFrame({"Distancia con area en m2": np.linalg.norm(original[1:] - original[0], axis=1), "Distancia con area en miles de m2": np.linalg.norm(otras_unidades[1:] - otras_unidades[0], axis=1)}, index=["A", "B"])
comparacion


**Discusión:** ¿Por qué cambia la alternativa más cercana? Escalar implica decidir cuánto pesa cada característica; no existe una escala universalmente correcta. En un flujo supervisado, los parámetros de escalado se aprenden solo de entrenamiento.


## Mahalanobis y covarianza
$d_M(x,y)=\sqrt{(x-y)^T S^{-1}(x-y)}$, con una matriz de covarianza invertible $S$. Consideremos $S=\begin{pmatrix}4&3\\3&4\end{pmatrix}$: las variables varían juntas. Dos desplazamientos pueden tener igual distancia euclidiana y distinta distancia de Mahalanobis.


In [ ]:
S = np.array([[4., 3.], [3., 4.]])
desplazamientos = np.array([[2., 2.], [2., -2.]])
resultados = []
for delta in desplazamientos:
    resultados.append({"Desplazamiento": str(delta), "Euclidiana": np.linalg.norm(delta), "Mahalanobis": np.sqrt(delta @ np.linalg.solve(S, delta))})
pd.DataFrame(resultados)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for delta, color in zip(desplazamientos, ["tab:blue", "tab:orange"]):
    ax.plot([0, delta[0]], [0, delta[1]], marker="o", color=color, label=str(delta))
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set(xlabel="Caracteristica 1", ylabel="Caracteristica 2", title="Desplazamientos desde el origen")
ax.set_aspect("equal")
ax.legend()
plt.show()


**Interpretación:** moverse en la dirección de variación conjunta recibe una distancia menor que moverse contra ella. Aquí la covarianza se proporciona para aislar el concepto. Al estimarla de datos, revisar su invertibilidad y aprenderla del entrenamiento si se evaluará un modelo supervisado.


## Ejercicios y cierre
1. Cambia los puntos y comprueba a mano un caso para Manhattan y euclidiana.
2. Cambia el exponente de Minkowski entre valores mayores o iguales a uno y explica el resultado.
3. Repite Mahalanobis con la matriz identidad: ¿con qué distancia coincide?
4. Propón dos características de tu conjunto de datos con escalas distintas y explica cómo podrían alterar un agrupamiento.
5. Explica por qué elegir una distancia requiere conocer las variables, sus unidades y el propósito del análisis.
